# NimbusPay Ticket Triage — LLM Fine-Tuning Assessment (VIMA3YA)

**Model:** `Qwen/Qwen2.5-3B-Instruct` (3.09B params ≤ 4B limit, ungated, native `system` role, separate `pad_token` / `eos_token`)  
**Stack:** Hugging Face `transformers` + `peft` + `trl` (`SFTTrainer` / `SFTConfig`) + `bitsandbytes`  
**Hardware target:** Google Colab free tier — NVIDIA T4 (16 GB VRAM)

This notebook executes end-to-end top-to-bottom:
- **Part A:** Data audit & deterministic cleaning (`train.jsonl` → `train_clean.jsonl` + `cleaning_log.csv`)
- **Part B:** Dev baselines (zero-shot fixed prompt vs. `SCHEMA.md` rules-in-prompt reference)
- **Part C:** Chat-template formatting, `-100` assistant-only loss masking, and evidence-based sequence length (`max_length = 2560`)
- **Part D:** Five controlled single-variable experiments (Raw vs. Cleaned, 4-bit QLoRA vs. 16-bit LoRA, Rank `r=8` vs. `r=16`, and `max_length=512` vs. `2560`)
- **Part E:** Best-model error analysis on `dev.jsonl` (per-field, per-category, and specific failures)
- **Submission:** Save final PEFT `adapter/`, generate `predictions_test.jsonl` (`batch_size=1` greedy), and run a fresh-load reproducibility check.


In [ ]:
# Install / upgrade core Hugging Face fine-tuning stack for Colab T4
import sys
if "google.colab" in sys.modules:
    !pip install -q -U transformers peft trl bitsandbytes accelerate datasets


In [ ]:
import csv
import gc
import inspect
import json
import os
import random
import re
import shutil
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
import peft
import trl
import bitsandbytes as bnb
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

SEED = 42
BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
SKIP_COMPLETED = True  # Set False to force retraining all experiments from scratch

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Python:", sys.version.split()[0])
print(f"torch={torch.__version__}, transformers={transformers.__version__}, peft={peft.__version__}, trl={trl.__version__}, bnb={bnb.__version__}")
assert torch.cuda.is_available(), "CUDA GPU is required! Switch Colab runtime to T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")


In [ ]:
# Locate candidate_pack (supports Google Drive, /content, or local repo) and configure persistent OUT_DIR
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

CANDIDATE_SEARCH_PATHS = [
    Path("/content/drive/MyDrive/vima3ya_assessment/candidate_pack"),
    Path("/content/drive/MyDrive/candidate_pack"),
    Path("/content/candidate_pack"),
    Path("./candidate_pack"),
    Path("../training_assessment/candidate_pack"),
]

PACK_DIR = next((p for p in CANDIDATE_SEARCH_PATHS if (p / "data" / "train.jsonl").exists()), None)
if PACK_DIR is None:
    # Auto-unzip if training_assessment.zip is uploaded to /content or MyDrive
    for zpath in [
        Path("/content/training_assessment.zip"),
        Path("/content/drive/MyDrive/vima3ya_assessment/training_assessment.zip"),
        Path("/content/drive/MyDrive/training_assessment.zip"),
    ]:
        if zpath.exists():
            import zipfile
            extract_dir = Path("/content/extracted_pack")
            with zipfile.ZipFile(zpath, "r") as zf:
                zf.extractall(extract_dir)
            found = list(extract_dir.rglob("train.jsonl"))
            if found:
                PACK_DIR = found[0].parent.parent
                break

assert PACK_DIR is not None, (
    "Could not find candidate_pack/data/train.jsonl. Upload candidate_pack/ or training_assessment.zip "
    "to /content/ or Google Drive (MyDrive/vima3ya_assessment/)."
)

if IN_COLAB and Path("/content/drive/MyDrive").exists():
    OUT_DIR = Path("/content/drive/MyDrive/vima3ya_assessment/outputs")
else:
    OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = OUT_DIR / "work_data"
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = PACK_DIR / "data"
SCHEMA_PATH = PACK_DIR / "SCHEMA.md"
print("PACK_DIR:", PACK_DIR)
print("OUT_DIR :", OUT_DIR)


In [ ]:
# Exact scoring logic from candidate_pack/score.py (plus non-leaked dev slice tracking)
KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]

def load_jsonl(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def save_jsonl(rows: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def _same(a, b) -> bool:
    return type(a) is type(b) and a == b

def _parse_json_strict(text):
    try:
        obj = json.loads(text.strip())
    except (json.JSONDecodeError, AttributeError):
        return None
    return obj if isinstance(obj, dict) else None

def score_predictions(gold_rows: list[dict], pred_rows: list[dict]) -> dict:
    preds = {p["id"]: p.get("output") for p in pred_rows}
    groups = defaultdict(lambda: {"n": 0, "valid": 0, "exact": 0, "field": defaultdict(int)})
    for g in gold_rows:
        gold = json.loads(g["messages"][-1]["content"])
        pred = _parse_json_strict(preds.get(g["id"]))
        names = ["all"] + ([f"slice={g['slice']}"] if "slice" in g else []) + [f"category={gold['category']}"]
        hits = {k: pred is not None and k in pred and _same(pred[k], gold[k]) for k in KEYS}
        exact = pred is not None and all(hits.values()) and set(pred) == set(KEYS)
        for name in names:
            s = groups[name]
            s["n"] += 1
            s["valid"] += int(pred is not None)
            s["exact"] += int(exact)
            for k in KEYS:
                s["field"][k] += int(hits[k])
    out = {}
    for name, s in groups.items():
        n = s["n"]
        fields = {k: s["field"][k] / n for k in KEYS}
        out[name] = {
            "n": n,
            "json_valid": s["valid"] / n,
            "exact_match": s["exact"] / n,
            "mean_field_acc": sum(fields.values()) / len(KEYS),
            "fields": fields,
        }
    return out

def print_score_report(res: dict, title: str = "Score Report"):
    print(f"=== {title} ===")
    print(f"{'group':28} {'n':>5} {'valid':>7} {'exact':>7} {'fields':>7}")
    for name in sorted(res, key=lambda x: (x != "all", x)):
        r = res[name]
        print(f"{name:28} {r['n']:5d} {r['json_valid']:7.1%} {r['exact_match']:7.1%} {r['mean_field_acc']:7.1%}")
    print("Per-field accuracy (all):")
    for k, v in res["all"]["fields"].items():
        print(f"  {k:14} {v:6.1%}")

raw_train_rows = load_jsonl(DATA_DIR / "train.jsonl")
dev_rows = load_jsonl(DATA_DIR / "dev.jsonl")
test_rows = load_jsonl(DATA_DIR / "test_inputs.jsonl")
print(f"Loaded: train={len(raw_train_rows)}, dev={len(dev_rows)}, test={len(test_rows)}")


## Part A: Data Audit & Cleaning (25 pts)

Auditing `train.jsonl` (1,445 rows) against `SCHEMA.md` and `dev.jsonl` reveals **355 defective rows** spanning 9 distinct real-world pipeline issues:
1. **Truncated / unparseable JSON labels (25 rows → `drop`):** Assistant output cut off mid-JSON (`Unterminated string` / missing trailing fields).
2. **Junk / empty / placeholder tickets (10 rows → `drop`):** Empty text or `"test"` / `"test test 123"` / `"asdf asdf"` paired with arbitrary labels.
3. **Dev-set contamination in `train.jsonl` (30 exact + 15 near-duplicate rows → `drop`):** Violates the strict *"Do not train on dev"* rule; perturbed near-duplicates differ only by extra spaces, trailing `" ."`, or `" Thx"`.
4. **In-train duplicates (60 exact + 40 near-duplicate rows → `drop`):** Overweights memorized tickets in SFT.
5. **Legacy v1 schema (`"prio"` + `DD/MM/YYYY`, 70 rows → `fix`):** All 70 rows use `"prio": "low"|"medium"|"high"` instead of `"priority": "P3"|"P2"|"P1"`, and 30 of them store `txn_date` as `DD/MM/YYYY` instead of `YYYY-MM-DD`.
6. **Corrupted `priority` under v2 rules (60 rows → `fix`):** `priority` label contradicts `SCHEMA.md` v2 priority rules (note that `needs_human` on those rows already matches the true `P1` rule).
7. **Rupees stored instead of paise (25 rows → `fix`):** `amount_paise` recorded in integer rupees instead of integer paise (`× 100`, including decimal truncation on 5 rows).
8. **Unnormalized `txn_id` (20 rows → `fix`):** Lowercase `np`, hyphens, or spaces instead of normalized `NP` + 10 digits.


In [ ]:
ALLOWED_CATEGORIES = {"payment_failed", "refund", "fraud", "account_access", "kyc", "offers", "other"}
ALLOWED_PRIORITIES = {"P1", "P2", "P3"}
ALLOWED_CHANNELS = {"upi", "card", "netbanking", "wallet", None}
ALLOWED_LANGUAGES = {"en", "hi-en"}
V1_PRIO_MAP = {"high": "P1", "medium": "P2", "low": "P3"}

LEGAL_PAT = re.compile(r"\b(rbi|ombudsman|consumer\s*court|lawyer|legal\s*notice|legal\s*action)\b", re.I)
AMT_PAT = re.compile(
    r"(?:(?:₹|rs\.?|inr)\s*([\d,]+(?:\.\d+)?)(?:\s*(k|lakh|lakhs)\b)?|"
    r"\b([\d,]+(?:\.\d+)?)\s*(rupees|rupaye|k|lakh|lakhs)\b)",
    re.I,
)

def normalize_for_dedup(user_text: str) -> str:
    s = re.sub(r"\s+", " ", user_text).strip()
    s = re.sub(r"\s+Thx$", "", s)
    s = re.sub(r"\s+\.$", "", s)
    return re.sub(r"\s+", " ", s).strip()

def non_quoted_body(user_text: str) -> str:
    body = user_text.split("\n\n", 1)[1] if "\n\n" in user_text else user_text
    return "\n".join(line for line in body.splitlines() if not line.lstrip().startswith(">"))

def extract_money_candidates(text: str):
    cands = []
    for m in AMT_PAT.finditer(text):
        g1, u1, g2, u2 = m.groups()
        num_str = (g1 or g2).replace(",", "")
        unit = (u1 or u2 or "").lower()
        try:
            val = float(num_str)
            if unit == "k":
                val *= 1000
            elif unit.startswith("lakh"):
                val *= 100000
            cands.append((val, int(round(val * 100)), int(val)))
        except ValueError:
            continue
    return cands

def compute_expected_priority(category: str, amount_paise, non_quoted: str) -> str:
    has_legal = bool(LEGAL_PAT.search(non_quoted))
    if category == "fraud" or (amount_paise is not None and amount_paise >= 5_000_000) or has_legal:
        return "P1"
    if (category in {"payment_failed", "refund"} and amount_paise is not None and amount_paise >= 500_000) or category == "account_access":
        return "P2"
    return "P3"

def clean_training_data(train_rows: list[dict], dev_rows: list[dict]):
    dev_exact = {r["messages"][1]["content"]: r["id"] for r in dev_rows}
    dev_near = {normalize_for_dedup(r["messages"][1]["content"]): r["id"] for r in dev_rows}

    seen_exact, seen_near = {}, {}
    log_entries = []
    cleaned_rows = []
    reason_counts = Counter()
    examples = {}
    leaked_dev_ids = set()

    for row in train_rows:
        rid = row["id"]
        msgs = row["messages"]
        user_text = msgs[1]["content"]
        raw_label = msgs[-1]["content"]
        body = (user_text.split("\n\n", 1)[1] if "\n\n" in user_text else user_text).strip()

        try:
            obj = json.loads(raw_label.strip())
            if not isinstance(obj, dict):
                raise ValueError("Not a JSON object")
        except Exception:
            log_entries.append((rid, "drop", "truncated_invalid_json"))
            reason_counts["drop:truncated_invalid_json"] += 1
            examples.setdefault("drop:truncated_invalid_json", (rid, raw_label))
            continue

        if len(body) < 20 or re.fullmatch(r"(test(\s+test)*(\s+\d+)?|asdf(\s+asdf)*)", body, re.I):
            log_entries.append((rid, "drop", f"junk_empty_ticket (body={body!r})"))
            reason_counts["drop:junk_empty_ticket"] += 1
            examples.setdefault("drop:junk_empty_ticket", (rid, body, raw_label[:80]))
            continue

        norm_u = normalize_for_dedup(user_text)
        if user_text in dev_exact:
            leaked_dev_ids.add(dev_exact[user_text])
            log_entries.append((rid, "drop", f"dev_leakage_exact:{dev_exact[user_text]}"))
            reason_counts["drop:dev_leakage_exact"] += 1
            examples.setdefault("drop:dev_leakage_exact", (rid, dev_exact[user_text]))
            continue
        if norm_u in dev_near:
            leaked_dev_ids.add(dev_near[norm_u])
            log_entries.append((rid, "drop", f"dev_leakage_near:{dev_near[norm_u]}"))
            reason_counts["drop:dev_leakage_near"] += 1
            examples.setdefault("drop:dev_leakage_near", (rid, dev_near[norm_u]))
            continue

        if user_text in seen_exact:
            log_entries.append((rid, "drop", f"exact_duplicate_of:{seen_exact[user_text]}"))
            reason_counts["drop:exact_duplicate"] += 1
            examples.setdefault("drop:exact_duplicate", (rid, seen_exact[user_text]))
            continue
        if norm_u in seen_near:
            log_entries.append((rid, "drop", f"near_duplicate_of:{seen_near[norm_u]}"))
            reason_counts["drop:near_duplicate"] += 1
            examples.setdefault("drop:near_duplicate", (rid, seen_near[norm_u]))
            continue
        seen_exact[user_text] = rid
        seen_near[norm_u] = rid

        fix_reasons = []
        if "prio" in obj:
            old_p = obj.pop("prio")
            obj["priority"] = V1_PRIO_MAP[old_p]
            fix_reasons.append(f"v1_prio_schema:{old_p}->{obj['priority']}")
            reason_counts["fix:v1_prio_schema"] += 1
            examples.setdefault("fix:v1_prio_schema", (rid, old_p, obj["priority"]))

        tdate = obj.get("txn_date")
        if tdate is not None and re.match(r"^\d{2}/\d{2}/\d{4}$", str(tdate)):
            d, m, y = tdate.split("/")
            obj["txn_date"] = f"{y}-{m}-{d}"
            fix_reasons.append(f"txn_date_ddmmyyyy:{tdate}->{obj['txn_date']}")
            reason_counts["fix:txn_date_ddmmyyyy"] += 1
            examples.setdefault("fix:txn_date_ddmmyyyy", (rid, tdate, obj["txn_date"]))

        tid = obj.get("txn_id")
        if tid is not None and not re.match(r"^NP\d{10}$", str(tid)):
            norm_tid = re.sub(r"[^A-Za-z0-9]", "", str(tid)).upper()
            assert re.match(r"^NP\d{10}$", norm_tid)
            fix_reasons.append(f"txn_id_unnormalized:{tid}->{norm_tid}")
            obj["txn_id"] = norm_tid
            reason_counts["fix:txn_id_unnormalized"] += 1
            examples.setdefault("fix:txn_id_unnormalized", (rid, tid, norm_tid))

        nq = non_quoted_body(user_text)
        cat = obj["category"]
        amt = obj.get("amount_paise")
        if cat not in {"account_access", "kyc", "other"} and amt is not None:
            cands = extract_money_candidates(nq)
            if all(paise != amt for _, paise, _ in cands):
                matched = [paise for _, paise, trunc_r in cands if trunc_r == amt]
                assert matched, f"{rid}: unexplained amt={amt}, cands={cands}"
                fix_reasons.append(f"amount_in_rupees:{amt}->{matched[0]}")
                obj["amount_paise"] = matched[0]
                reason_counts["fix:amount_in_rupees"] += 1
                examples.setdefault("fix:amount_in_rupees", (rid, amt, matched[0]))

        exp_prio = compute_expected_priority(cat, obj.get("amount_paise"), nq)
        if obj["priority"] != exp_prio:
            old_prio = obj["priority"]
            fix_reasons.append(f"priority_v2_rule:{old_prio}->{exp_prio}")
            obj["priority"] = exp_prio
            reason_counts["fix:priority_v2_rule"] += 1
            examples.setdefault("fix:priority_v2_rule", (rid, f"{old_prio}->{exp_prio}"))

        clean_obj = {k: obj[k] for k in KEYS}
        if fix_reasons:
            log_entries.append((rid, "fix", "; ".join(fix_reasons)))

        cleaned_rows.append({
            "id": rid,
            "messages": [msgs[0], msgs[1], {"role": "assistant", "content": json.dumps(clean_obj, ensure_ascii=False)}],
        })

    return cleaned_rows, log_entries, reason_counts, examples, leaked_dev_ids

clean_train_rows, log_entries, reason_counts, audit_examples, LEAKED_DEV_IDS = clean_training_data(raw_train_rows, dev_rows)

CLEAN_TRAIN_PATH = WORK_DIR / "train_clean.jsonl"
CLEANING_LOG_PATH = OUT_DIR / "cleaning_log.csv"
save_jsonl(clean_train_rows, CLEAN_TRAIN_PATH)
with open(CLEANING_LOG_PATH, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "action", "reason"])
    w.writerows(log_entries)

# Strict integrity assertions
assert len({r["messages"][1]["content"] for r in clean_train_rows} & {r["messages"][1]["content"] for r in dev_rows}) == 0
assert len({r["messages"][1]["content"] for r in clean_train_rows} & {r["messages"][1]["content"] for r in test_rows}) == 0
self_score = score_predictions(clean_train_rows, [{"id": r["id"], "output": r["messages"][-1]["content"]} for r in clean_train_rows])
assert self_score["all"]["exact_match"] == 1.0, f"Cleaned data failed self-score: {self_score['all']}"

print(f"Raw train: {len(raw_train_rows)} -> Cleaned train: {len(clean_train_rows)}")
print(f"Logged actions: {len(log_entries)} ({sum(a=='drop' for _,a,_ in log_entries)} drop, {sum(a=='fix' for _,a,_ in log_entries)} fix)")
print(f"Leaked dev rows caught & excluded from train: {len(LEAKED_DEV_IDS)} (uncontaminated dev rows = {len(dev_rows) - len(LEAKED_DEV_IDS)})")
for k, v in sorted(reason_counts.items()):
    print(f"  {k:32s}: {v:3d}  | sample={audit_examples[k]}")


In [ ]:
# Visualize Part A Audit Breakdown & Label Distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
items = sorted(reason_counts.items(), key=lambda x: x[1])
labels_bar = [k for k, _ in items]
vals_bar = [v for _, v in items]
colors = ["#d9534f" if k.startswith("drop:") else "#f0ad4e" for k in labels_bar]
axes[0].barh(labels_bar, vals_bar, color=colors)
axes[0].set_title(f"Part A Audit Issues Found in train.jsonl ({len(log_entries)} rows affected)")
axes[0].set_xlabel("Row count")

cat_counts = Counter(json.loads(r["messages"][-1]["content"])["category"] for r in clean_train_rows)
axes[1].bar(cat_counts.keys(), cat_counts.values(), color="#3b82f6")
axes[1].set_title(f"Cleaned Training Category Distribution (n={len(clean_train_rows)})")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.savefig(OUT_DIR / "part_a_audit_summary.png", dpi=150)
plt.show()


## Part C (Prep): Chat Template, Assistant-Only Loss Masking & Token Length Evidence

Before running baselines or fine-tuning, we inspect exact token counts with `Qwen/Qwen2.5-3B-Instruct`'s tokenizer:
- 97% of tickets are ≤ 265 tokens, **but** long email threads containing `>` quoted support reply chains reach **2,215 tokens** (40 in train, 3 in dev, 11 in test).
- Choosing `max_length = 2560` keeps **100%** of long quoted-email training rows intact so the model learns to ignore `>` quoted dates (`SCHEMA.md`).
- Setting `labels[:prompt_len] = -100` ensures cross-entropy loss is computed **only** on the assistant JSON + `<|im_end|>` tokens.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.padding_side = "right"
assert tokenizer.pad_token_id is not None and tokenizer.pad_token_id != tokenizer.eos_token_id, "Expected distinct pad and eos tokens"
print(f"Tokenizer loaded: pad_token={tokenizer.pad_token!r} ({tokenizer.pad_token_id}), eos_token={tokenizer.eos_token!r} ({tokenizer.eos_token_id})")

def measure_token_lengths(rows, is_test=False):
    lens = []
    for r in rows:
        msgs = r["messages"]
        txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=is_test)
        lens.append(len(tokenizer.encode(txt, add_special_tokens=False)))
    return sorted(lens)

tr_lens = measure_token_lengths(clean_train_rows, is_test=False)
dv_lens = measure_token_lengths(dev_rows, is_test=False)
ts_lens = measure_token_lengths(test_rows, is_test=True)

for name, arr in [("train_clean", tr_lens), ("dev", dv_lens), ("test_prompt", ts_lens)]:
    n = len(arr)
    print(f"{name:12s}: n={n:4d}, min={arr[0]:3d}, p50={arr[n//2]:3d}, p95={arr[int(n*0.95)]:3d}, p97={arr[int(n*0.97)]:3d}, max={arr[-1]:4d}, >512={sum(x>512 for x in arr)}")

MAX_LENGTH_EVIDENCE = 2560
assert tr_lens[-1] <= MAX_LENGTH_EVIDENCE and dv_lens[-1] <= MAX_LENGTH_EVIDENCE

plt.figure(figsize=(8, 3.5))
plt.hist(tr_lens, bins=40, color="#2563eb", alpha=0.85, edgecolor="white")
plt.axvline(512, color="red", linestyle="--", label="Naive cutoff (512) -> truncates 30+ quoted emails")
plt.axvline(MAX_LENGTH_EVIDENCE, color="green", linestyle="-", label=f"Chosen max_length ({MAX_LENGTH_EVIDENCE}) -> 0% truncation")
plt.title("Token Length Distribution of Cleaned Training Examples (Qwen2.5 Tokenizer)")
plt.xlabel("Total tokens (system + user + assistant JSON)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "token_length_distribution.png", dpi=150)
plt.show()


In [ ]:
def prepare_tokenized_dataset(rows: list[dict], tokenizer, max_length: int = MAX_LENGTH_EVIDENCE, drop_if_no_target: bool = True):
    records = []
    dropped_truncated = 0
    for r in rows:
        msgs = r["messages"]
        prompt_text = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
        full_ids = tokenizer.encode(full_text, add_special_tokens=False)
        assert full_ids[:len(prompt_ids)] == prompt_ids, f"Prefix mismatch on {r['id']}"

        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
        if len(full_ids) > max_length:
            full_ids = full_ids[:max_length]
            labels = labels[:max_length]
            if drop_if_no_target and all(x == -100 for x in labels):
                dropped_truncated += 1
                continue

        records.append({
            "input_ids": full_ids,
            "attention_mask": [1] * len(full_ids),
            "labels": labels,
        })
    return Dataset.from_list(records), dropped_truncated

class CompletionPaddingCollator:
    """Dynamic right-padding collator preserving -100 prompt masks."""
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch_ids, batch_mask, batch_labels = [], [], []
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch_ids.append(f["input_ids"] + [self.pad_token_id] * pad_len)
            batch_mask.append(f["attention_mask"] + [0] * pad_len)
            batch_labels.append(f["labels"] + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(batch_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }

# Sanity check masking on clean_train_rows
ds_check, dropped_2560 = prepare_tokenized_dataset(clean_train_rows, tokenizer, max_length=MAX_LENGTH_EVIDENCE)
assert dropped_2560 == 0
sample0 = ds_check[0]
unmasked = [tid for tid in sample0["labels"] if tid != -100]
print(f"Prepared {len(ds_check)} rows (dropped={dropped_2560}).")
print(f"Row 0: total={len(sample0['input_ids'])}, masked(-100)={len(sample0['input_ids'])-len(unmasked)}, target_tokens={len(unmasked)}")
print("Decoded unmasked target:", repr(tokenizer.decode(unmasked)))


## Part B: Baselines on Dev (10 pts)

We evaluate the base `Qwen/Qwen2.5-3B-Instruct` model (no fine-tuning) on `dev.jsonl` using greedy decoding (`max_new_tokens=200`, whitespace `.strip()` only):
1. **Baseline 1 (Fixed prompt):** Exact system prompt from the data, no rules.
2. **Baseline 2 (Rules-in-prompt reference):** Same system prompt augmented with `SCHEMA.md` rules (allowed here only as a reference point).


In [ ]:
def run_greedy_inference(model, tokenizer, rows: list[dict], batch_size: int = 8, system_override: str | None = None, max_new_tokens: int = 200) -> list[dict]:
    """Run greedy model.generate matching ASSIGNMENT.md rules (strip whitespace only)."""
    model.eval()
    orig_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    prompts = []
    for r in rows:
        msgs = [{"role": m["role"], "content": m["content"]} for m in r["messages"][:2]]
        if system_override is not None:
            msgs[0]["content"] = system_override
        prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

    preds = []
    try:
        for i in range(0, len(rows), batch_size):
            batch_prompts = prompts[i : i + batch_size]
            enc = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(model.device)
            prompt_width = enc["input_ids"].shape[1]
            with torch.inference_mode():
                gen_ids = model.generate(
                    **enc,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    temperature=None,
                    top_p=None,
                    top_k=None,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    use_cache=True,
                )
            for j in range(len(batch_prompts)):
                out_text = tokenizer.decode(gen_ids[j, prompt_width:], skip_special_tokens=True).strip()
                preds.append({"id": rows[i + j]["id"], "output": out_text})
    finally:
        tokenizer.padding_side = orig_side
    return preds

def load_base_model(use_4bit: bool = True):
    dtype_kw = {"dtype": torch.float16} if int(transformers.__version__.split(".")[0]) >= 5 else {"torch_dtype": torch.float16}
    if use_4bit:
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        return AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb_cfg, device_map="auto", **dtype_kw)
    return AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, device_map="auto", **dtype_kw)

def free_cuda(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
baseline_dir = OUT_DIR / "baselines"
baseline_dir.mkdir(parents=True, exist_ok=True)
b1_path = baseline_dir / "baseline1_fixed_prompt.json"
b2_path = baseline_dir / "baseline2_rules_in_prompt.json"

schema_text = SCHEMA_PATH.read_text(encoding="utf-8")
RULES_SYSTEM_PROMPT = (
    "You are NimbusPay's ticket triage model. Follow this exact labelling specification and return ONLY a single valid JSON object "
    "(no markdown fences, no commentary).\n\n" + schema_text
)

if SKIP_COMPLETED and b1_path.exists() and b2_path.exists():
    b1_res = json.loads(b1_path.read_text(encoding="utf-8"))
    b2_res = json.loads(b2_path.read_text(encoding="utf-8"))
    print("Loaded saved baseline metrics.")
else:
    base_model = load_base_model(use_4bit=False)

    t0 = time.time()
    b1_preds = run_greedy_inference(base_model, tokenizer, dev_rows, batch_size=8)
    save_jsonl(b1_preds, baseline_dir / "baseline1_dev_preds.jsonl")
    b1_res = score_predictions(dev_rows, b1_preds)
    b1_path.write_text(json.dumps(b1_res, indent=2), encoding="utf-8")
    print(f"Baseline 1 completed in {(time.time()-t0)/60:.2f} min")

    t0 = time.time()
    b2_preds = run_greedy_inference(base_model, tokenizer, dev_rows, batch_size=8, system_override=RULES_SYSTEM_PROMPT)
    save_jsonl(b2_preds, baseline_dir / "baseline2_dev_preds.jsonl")
    b2_res = score_predictions(dev_rows, b2_preds)
    b2_path.write_text(json.dumps(b2_res, indent=2), encoding="utf-8")
    print(f"Baseline 2 completed in {(time.time()-t0)/60:.2f} min")

    free_cuda(base_model)

print_score_report(b1_res, "Baseline 1: Zero-Shot Fixed Prompt")
print()
print_score_report(b2_res, "Baseline 2: Rules-in-Prompt (Reference)")


## Part C & D: SFT Pipeline & Controlled Experiments (40 pts)

All experiments vary **exactly one hyperparameter or dataset condition** at a time around our anchor (`Exp 2`):
1. **`exp1_raw_qlora`**: Raw `train.jsonl` (1,445 rows) vs. Cleaned `train_clean.jsonl` (1,265 rows) — *Data quality ablation*
2. **`exp2_clean_qlora`** *(Anchor)*: Cleaned data, 4-bit NF4 QLoRA, `r=16, alpha=32`, all linear projections, `lr=2e-4`, 3 epochs, `max_length=2560`
3. **`exp3_clean_fp16_lora`**: 16-bit `fp16` LoRA vs. 4-bit NF4 QLoRA — *Quantization / precision ablation*
4. **`exp4_clean_qlora_r8`**: LoRA rank `r=8, alpha=16` vs. `r=16, alpha=32` — *Adapter capacity sweep*
5. **`exp5_clean_qlora_len512`**: Naive `max_length=512` (drops long quoted-email targets) vs. evidence-based `max_length=2560` — *Own ablation idea*

We also record the **uncontaminated 155-row dev score** (excluding the 45 dev rows that were leaked into raw `train.jsonl`) alongside the full 200-row dev score.


In [ ]:
ALL_LINEAR = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
uncontaminated_dev_rows = [r for r in dev_rows if r["id"] not in LEAKED_DEV_IDS]

def run_experiment(
    exp_name: str,
    description: str,
    train_rows: list[dict],
    *,
    use_4bit: bool = True,
    lora_r: int = 16,
    lora_alpha: int = 32,
    target_modules: list[str] = ALL_LINEAR,
    lr: float = 2e-4,
    epochs: int = 3,
    max_length: int = MAX_LENGTH_EVIDENCE,
    seed: int = SEED,
) -> dict:
    exp_dir = OUT_DIR / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = exp_dir / "metrics.json"
    adapter_dir = exp_dir / "adapter"

    if SKIP_COMPLETED and metrics_path.exists() and (adapter_dir / "adapter_config.json").exists():
        print(f"[SKIP] Loaded saved metrics for {exp_name}")
        return json.loads(metrics_path.read_text(encoding="utf-8"))

    set_seed(seed)
    train_ds, dropped_long = prepare_tokenized_dataset(train_rows, tokenizer, max_length=max_length, drop_if_no_target=True)
    eval_ds, _ = prepare_tokenized_dataset(dev_rows, tokenizer, max_length=max_length, drop_if_no_target=True)
    print(f"\n>>> Running {exp_name}: {description} | train_n={len(train_ds)} (dropped_long={dropped_long})")

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    model = load_base_model(use_4bit=use_4bit)
    if use_4bit:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.enable_input_require_grads()

    lora_cfg = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=target_modules,
    )
    model = get_peft_model(model, lora_cfg)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    assert trainable_params <= 50_000_000, f"Exceeded 50M trainable param limit: {trainable_params}"
    print(f"Trainable params: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

    sft_kwargs = dict(
        output_dir=str(exp_dir / "hf_ckpts"),
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=lr,
        num_train_epochs=epochs,
        lr_scheduler_type="cosine",
        warmup_steps=15,
        fp16=True,
        bf16=False,
        optim="adamw_torch",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=20,
        remove_unused_columns=False,
        report_to="none",
        seed=seed,
    )
    sft_sig = inspect.signature(SFTConfig.__init__).parameters
    has_var_kw = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sft_sig.values())
    if "max_length" in sft_sig:
        sft_kwargs["max_length"] = max_length
    elif "max_seq_length" in sft_sig:
        sft_kwargs["max_seq_length"] = max_length
    if "dataset_kwargs" in sft_sig:
        sft_kwargs["dataset_kwargs"] = {"skip_prepare_dataset": True}

    if not has_var_kw:
        sft_kwargs = {k: v for k, v in sft_kwargs.items() if k in sft_sig}

    trainer_kwargs = dict(
        model=model,
        args=SFTConfig(**sft_kwargs),
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=CompletionPaddingCollator(tokenizer.pad_token_id),
    )
    tr_sig = inspect.signature(SFTTrainer.__init__).parameters
    if "processing_class" in tr_sig:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = SFTTrainer(**trainer_kwargs)
    trainer.train()

    train_minutes = (time.time() - t0) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    model.save_pretrained(adapter_dir)
    log_hist_path = exp_dir / "log_history.json"
    log_hist_path.write_text(json.dumps(trainer.state.log_history, indent=2), encoding="utf-8")

    # Evaluate on dev
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    model.config.use_cache = True
    dev_preds = run_greedy_inference(model, tokenizer, dev_rows, batch_size=8)
    save_jsonl(dev_preds, exp_dir / "dev_preds.jsonl")
    dev_score = score_predictions(dev_rows, dev_preds)
    dev_uncontam_score = score_predictions(uncontaminated_dev_rows, dev_preds)

    summary = {
        "exp_name": exp_name,
        "description": description,
        "use_4bit": use_4bit,
        "lora_r": lora_r,
        "lora_alpha": lora_alpha,
        "lr": lr,
        "epochs": epochs,
        "max_length": max_length,
        "train_rows": len(train_ds),
        "dropped_long": dropped_long,
        "trainable_params": trainable_params,
        "dev_json_valid": dev_score["all"]["json_valid"],
        "dev_exact_match": dev_score["all"]["exact_match"],
        "dev_mean_field_acc": dev_score["all"]["mean_field_acc"],
        "dev_uncontam155_exact": dev_uncontam_score["all"]["exact_match"],
        "peak_vram_gb": round(peak_vram_gb, 2),
        "train_minutes": round(train_minutes, 2),
        "full_dev_score": dev_score,
    }
    metrics_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(
        f"[DONE] {exp_name}: exact={summary['dev_exact_match']:.1%}, fields={summary['dev_mean_field_acc']:.1%}, "
        f"uncontam155={summary['dev_uncontam155_exact']:.1%}, vram={summary['peak_vram_gb']:.2f}GB, time={summary['train_minutes']:.2f}m"
    )

    free_cuda(trainer, model)
    return summary


In [ ]:
experiments = [
    ("exp1_raw_qlora", "Raw train.jsonl (1,445 rows), 4-bit QLoRA r=16", raw_train_rows, dict(use_4bit=True, lora_r=16, lora_alpha=32, max_length=2560)),
    ("exp2_clean_qlora", "Cleaned train_clean.jsonl (1,265 rows), 4-bit QLoRA r=16 [Anchor]", clean_train_rows, dict(use_4bit=True, lora_r=16, lora_alpha=32, max_length=2560)),
    ("exp3_clean_fp16_lora", "16-bit fp16 LoRA r=16 (vs 4-bit QLoRA)", clean_train_rows, dict(use_4bit=False, lora_r=16, lora_alpha=32, max_length=2560)),
    ("exp4_clean_qlora_r8", "4-bit QLoRA r=8, alpha=16 (vs r=16, alpha=32)", clean_train_rows, dict(use_4bit=True, lora_r=8, lora_alpha=16, max_length=2560)),
    ("exp5_clean_qlora_len512", "4-bit QLoRA max_length=512 (vs evidence max_length=2560)", clean_train_rows, dict(use_4bit=True, lora_r=16, lora_alpha=32, max_length=512)),
]

all_results = []
for name, desc, rows_in, kw in experiments:
    res = run_experiment(name, desc, rows_in, **kw)
    all_results.append(res)

exp_df = pd.DataFrame([
    {
        "Experiment": r["exp_name"],
        "What Changed": r["description"],
        "Trainable Params": f"{r['trainable_params']/1e6:.2f}M",
        "Dev JSON Valid": f"{r['dev_json_valid']:.1%}",
        "Dev Exact Match": f"{r['dev_exact_match']:.1%}",
        "Dev Mean Field Acc": f"{r['dev_mean_field_acc']:.1%}",
        "Dev Uncontam-155 Exact": f"{r['dev_uncontam155_exact']:.1%}",
        "Peak GPU Mem (GB)": r["peak_vram_gb"],
        "Train Min": r["train_minutes"],
    }
    for r in all_results
])
exp_df.to_csv(OUT_DIR / "experiments_table.csv", index=False)
exp_df


In [ ]:
# Plot train vs eval loss curves for the cleaned experiments
plt.figure(figsize=(9, 4.2))
for r in all_results:
    hist = json.loads((OUT_DIR / r["exp_name"] / "log_history.json").read_text(encoding="utf-8"))
    tr_pts = [(x["epoch"], x["loss"]) for x in hist if "loss" in x]
    ev_pts = [(x["epoch"], x["eval_loss"]) for x in hist if "eval_loss" in x]
    if tr_pts:
        plt.plot([p[0] for p in tr_pts], [p[1] for p in tr_pts], label=f"{r['exp_name']} (train)", alpha=0.8)
    if ev_pts:
        plt.plot([p[0] for p in ev_pts], [p[1] for p in ev_pts], "--o", label=f"{r['exp_name']} (dev)")
plt.title("Training & Dev Loss Dynamics Across Controlled Experiments")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(OUT_DIR / "loss_curves.png", dpi=150)
plt.show()

# Select best model strictly among CLEANED-data runs (never exp1_raw_qlora)
clean_runs = [r for r in all_results if r["exp_name"] != "exp1_raw_qlora"]
best_run = max(clean_runs, key=lambda r: (r["dev_exact_match"], r["dev_mean_field_acc"], -r["peak_vram_gb"]))
print(f"Selected BEST model: {best_run['exp_name']} | exact={best_run['dev_exact_match']:.1%}, fields={best_run['dev_mean_field_acc']:.1%}, 4bit={best_run['use_4bit']}")


## Part E: Error Analysis on Best Model (15 pts)

We inspect per-category and per-field accuracy on `dev.jsonl` for the winning model and dump concrete failure cases with gold vs. predicted fields.


In [ ]:
best_exp_dir = OUT_DIR / best_run["exp_name"]
best_dev_preds = load_jsonl(best_exp_dir / "dev_preds.jsonl")
best_dev_score = score_predictions(dev_rows, best_dev_preds)
print_score_report(best_dev_score, f"Best Model on Dev: {best_run['exp_name']}")

pred_map = {p["id"]: p["output"] for p in best_dev_preds}
failures = []
field_error_counts = Counter()

for g in dev_rows:
    gold = json.loads(g["messages"][-1]["content"])
    raw_out = pred_map.get(g["id"], "")
    pred = _parse_json_strict(raw_out)
    if pred is None:
        failures.append({"id": g["id"], "error_type": "invalid_json", "mismatches": {"json": (gold, raw_out)}, "ticket": non_quoted_body(g["messages"][1]["content"])})
        continue
    diffs = {}
    for k in KEYS:
        if k not in pred or not _same(pred[k], gold[k]):
            diffs[k] = {"gold": gold[k], "pred": pred.get(k, "<MISSING>")}
            field_error_counts[k] += 1
    if diffs or set(pred) != set(KEYS):
        failures.append({
            "id": g["id"],
            "category": gold["category"],
            "diffs": diffs,
            "extra_keys": sorted(set(pred) - set(KEYS)),
            "received": g["messages"][1]["content"].splitlines()[0],
            "ticket": non_quoted_body(g["messages"][1]["content"]),
            "has_quoted": ">" in g["messages"][1]["content"],
        })

(OUT_DIR / "error_analysis_failures.json").write_text(json.dumps(failures, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nTotal dev failures: {len(failures)} / {len(dev_rows)}")
print("Errors by field:", dict(field_error_counts.most_common()))
print("\n--- First 15 Specific Failure Cases ---")
for idx, f_item in enumerate(failures[:15], 1):
    print(f"\n[{idx}] {f_item['id']} | cat={f_item.get('category')} | {f_item.get('received', '')} | quoted={f_item.get('has_quoted')}")
    print("  Diffs:", json.dumps(f_item.get("diffs") or f_item.get("mismatches"), ensure_ascii=False))
    print("  Text :", f_item["ticket"].replace("\n", " ")[:240])


## Final Submission: Export `adapter/`, Generate `predictions_test.jsonl`, and Reproducibility Check

To guarantee **100% exact bit-for-bit match** with VIMA3YA's single-example `model.generate(..., do_sample=False, max_new_tokens=200)` evaluation check, we generate the 400 test predictions with `batch_size=1` (no left-padding) from the winning adapter, copy the adapter to `OUT_DIR / "final_submission"`, and verify a fresh reload of `PeftModel.from_pretrained` reproduces a random sample identically.


In [ ]:
SUBMISSION_DIR = OUT_DIR / "final_submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER_DIR = SUBMISSION_DIR / "adapter"
if FINAL_ADAPTER_DIR.exists():
    shutil.rmtree(FINAL_ADAPTER_DIR)
shutil.copytree(best_exp_dir / "adapter", FINAL_ADAPTER_DIR)
shutil.copy2(CLEANING_LOG_PATH, SUBMISSION_DIR / "cleaning_log.csv")

adapter_weights_size_mb = (FINAL_ADAPTER_DIR / "adapter_model.safetensors").stat().st_size / (1024 ** 2)
print(f"Copied final adapter -> {FINAL_ADAPTER_DIR} (weights = {adapter_weights_size_mb:.2f} MB < 100 MB limit)")

# Reload base model + final adapter cleanly and generate predictions_test.jsonl at batch_size=1
base_for_test = load_base_model(use_4bit=best_run["use_4bit"])
peft_test_model = PeftModel.from_pretrained(base_for_test, FINAL_ADAPTER_DIR)
peft_test_model.eval()

test_preds_path = SUBMISSION_DIR / "predictions_test.jsonl"
if SKIP_COMPLETED and test_preds_path.exists() and len(load_jsonl(test_preds_path)) == len(test_rows):
    test_preds = load_jsonl(test_preds_path)
    print(f"Loaded existing {test_preds_path} ({len(test_preds)} rows)")
else:
    t0 = time.time()
    test_preds = run_greedy_inference(peft_test_model, tokenizer, test_rows, batch_size=1, max_new_tokens=200)
    save_jsonl(test_preds, test_preds_path)
    print(f"Generated 400 test predictions (batch_size=1) in {(time.time()-t0)/60:.2f} min")

# Validate test prediction formatting
valid_json_cnt = 0
schema_ok_cnt = 0
for p in test_preds:
    obj = _parse_json_strict(p["output"])
    if obj is not None:
        valid_json_cnt += 1
        if set(obj.keys()) == set(KEYS):
            schema_ok_cnt += 1
print(f"Test predictions summary: total={len(test_preds)}, valid_json={valid_json_cnt}/{len(test_preds)}, exact_8_keys={schema_ok_cnt}/{len(test_preds)}")
assert len(test_preds) == len(test_rows) and [p["id"] for p in test_preds] == [r["id"] for r in test_rows]

free_cuda(peft_test_model, base_for_test)

# Independent reproducibility check (fresh PeftModel reload on 10 random test rows)
rng = random.Random(SEED)
sample_indices = sorted(rng.sample(range(len(test_rows)), 10))
sample_rows = [test_rows[i] for i in sample_indices]

repro_base = load_base_model(use_4bit=best_run["use_4bit"])
repro_model = PeftModel.from_pretrained(repro_base, FINAL_ADAPTER_DIR)
repro_preds = run_greedy_inference(repro_model, tokenizer, sample_rows, batch_size=1, max_new_tokens=200)
for idx, rp in zip(sample_indices, repro_preds):
    assert rp["output"] == test_preds[idx]["output"], f"Repro mismatch on {rp['id']}"
print("Reproducibility check PASSED: 10/10 fresh-reloaded predictions match predictions_test.jsonl identically!")
free_cuda(repro_model, repro_base)

# Save full results bundle for report generation
bundle = {
    "base_model": BASE_MODEL_ID,
    "baseline1": b1_res,
    "baseline2": b2_res,
    "experiments": [{k: v for k, v in r.items() if k != "full_dev_score"} for r in all_results],
    "best_run_name": best_run["exp_name"],
    "best_dev_score": best_dev_score,
    "test_summary": {"total": len(test_preds), "valid_json": valid_json_cnt, "exact_8_keys": schema_ok_cnt},
    "adapter_size_mb": round(adapter_weights_size_mb, 2),
}
(SUBMISSION_DIR / "results_bundle.json").write_text(json.dumps(bundle, indent=2), encoding="utf-8")
print("Saved complete results bundle to:", SUBMISSION_DIR / "results_bundle.json")
